<a href="https://colab.research.google.com/github/8009678200/ML.github.io/blob/main/hd12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Streamlit and pyngrok
!{sys.executable} -m pip install streamlit pyngrok

/bin/bash: line 1: {sys.executable}: command not found


Next, I'll create the `app.py` file which contains the Streamlit application code. This code includes a function to manually parse the CSV file (without using pandas), and then implements the logic for sorting and pagination, and finally displays the data using `st.table`.

In [2]:
%%writefile app.py

import streamlit as st
import csv
import math

# Function to manually parse CSV without pandas
def parse_csv_data(filepath):
    data = []
    headers = []
    try:
        with open(filepath, 'r', newline='', encoding='utf-8') as f:
            reader = csv.reader(f)
            headers = next(reader) # First row is headers
            for row in reader:
                if row: # Ensure row is not empty
                    data.append(row)
    except FileNotFoundError:
        st.error(f"File not found: {filepath}")
        return [], []
    except Exception as e:
        st.error(f"Error parsing CSV: {e}")
        return [], []
    return headers, data

# Function to convert specified categorical columns to numerical
def convert_categorical_to_numerical(headers, data, categorical_cols):
    # Create a copy to avoid modifying original data during iteration issues
    processed_data = [list(row) for row in data]

    for col_name in categorical_cols:
        try:
            col_idx = headers.index(col_name)
            unique_values = sorted(list(set(row[col_idx] for row in processed_data if len(row) > col_idx)))
            mapping = {value: i for i, value in enumerate(unique_values)}

            for i, row in enumerate(processed_data):
                if len(row) > col_idx:
                    original_value = row[col_idx]
                    # Assign numerical value; handle unknown values by assigning -1 or a unique new number
                    processed_data[i][col_idx] = mapping.get(original_value, -1) # -1 for unknown values
        except ValueError:
            # Column not found in headers, skip conversion for this column
            st.warning(f"Categorical column '{col_name}' not found in dataset headers. Skipping conversion.")
    return processed_data

# Function to calculate mean
def calculate_mean(data_list):
    if not data_list:
        return None
    total = 0
    count = 0
    for x in data_list:
        try:
            total += float(x)
            count += 1
        except ValueError:
            continue # Skip non-numeric values
    return total / count if count > 0 else None

# Function to calculate median
def calculate_median(data_list):
    numeric_data = []
    for x in data_list:
        try:
            numeric_data.append(float(x))
        except ValueError:
            continue
    if not numeric_data:
        return None
    sorted_data = sorted(numeric_data)
    n = len(sorted_data)
    if n % 2 == 1:
        return sorted_data[n // 2]
    else:
        mid1 = sorted_data[n // 2 - 1]
        mid2 = sorted_data[n // 2]
        return (mid1 + mid2) / 2

# Function to calculate mode
def calculate_mode(data_list):
    counts = {}
    for x in data_list:
        # Mode can be calculated for both numeric and non-numeric
        counts[x] = counts.get(x, 0) + 1
    if not counts:
        return None
    max_count = 0
    modes = []
    for value, count in counts.items():
        if count > max_count:
            max_count = count
            modes = [value]
        elif count == max_count and value not in modes:
            modes.append(value)
    return modes # Return a list because there can be multiple modes

# Function to calculate min
def calculate_min(data_list):
    if not data_list:
        return None
    min_val = None
    for x in data_list:
        try:
            val = float(x)
            if min_val is None or val < min_val:
                min_val = val
        except ValueError:
            continue
    return min_val

# Function to calculate max
def calculate_max(data_list):
    if not data_list:
        return None
    max_val = None
    for x in data_list:
        try:
            val = float(x)
            if max_val is None or val > max_val:
                max_val = val
        except ValueError:
            continue
    return max_val

# Function to calculate standard deviation (sample standard deviation)
def calculate_std_dev(data_list):
    numeric_data = []
    for x in data_list:
        try:
            numeric_data.append(float(x))
        except ValueError:
            continue
    if len(numeric_data) < 2: # Need at least two points for sample std dev
        return 0.0 if len(numeric_data) == 1 else None

    mean_val = calculate_mean(numeric_data)
    if mean_val is None:
        return None

    sum_sq_diff = 0
    for x in numeric_data:
        sum_sq_diff += (x - mean_val) ** 2

    # Using sample standard deviation (n-1 degrees of freedom)
    return math.sqrt(sum_sq_diff / (len(numeric_data) - 1))

# Function to identify missing values in processed_data
def identify_missing_values_in_processed_data(headers, data, categorical_cols_converted):
    missing_counts = {header: 0 for header in headers}
    missing_indices = {header: [] for header in headers}
    for row_idx, row in enumerate(data):
        for col_idx, value in enumerate(row):
            col_name = headers[col_idx]
            is_missing = False
            if isinstance(value, str) and (not value or value.strip() == ''): # Empty string for numerical columns
                is_missing = True
            elif col_name in categorical_cols_converted and value == -1: # -1 for converted categorical missing
                is_missing = True

            if is_missing:
                missing_counts[col_name] += 1
                missing_indices[col_name].append(row_idx)
    return missing_counts, missing_indices

# Helper to get numeric data from a column for imputation (from processed_data)
def get_column_numeric_values_for_imputation(data, col_idx, is_categorical_converted=False):
    numeric_data = []
    for row in data:
        if len(row) > col_idx:
            val = row[col_idx]
            if is_categorical_converted and val == -1: # Treat -1 as missing for converted categoricals
                continue
            if isinstance(val, str) and (not val or val.strip() == ''): # Treat empty string as missing
                continue

            try:
                numeric_data.append(float(val))
            except ValueError:
                continue # Skip truly non-numeric values
    return numeric_data

# Function to impute with mean (on processed_data)
def impute_with_mean(data, col_idx, is_categorical_converted=False):
    imputed_data = [list(row) for row in data]
    numeric_values = get_column_numeric_values_for_imputation(imputed_data, col_idx, is_categorical_converted)
    mean_val = calculate_mean(numeric_values)
    if mean_val is None:
        return imputed_data

    for r_idx, row in enumerate(imputed_data):
        if len(row) > col_idx:
            val = row[col_idx]
            is_missing = False
            if is_categorical_converted and val == -1:
                is_missing = True
            elif isinstance(val, str) and (not val or val.strip() == ''): # Check for empty string
                is_missing = True

            if is_missing:
                # Impute with rounded integer if it was a converted categorical, otherwise as float string
                imputed_data[r_idx][col_idx] = round(mean_val) if is_categorical_converted else f"{mean_val:.2f}"
    return imputed_data

# Function to impute with median (on processed_data)
def impute_with_median(data, col_idx, is_categorical_converted=False):
    imputed_data = [list(row) for row in data]
    numeric_values = get_column_numeric_values_for_imputation(imputed_data, col_idx, is_categorical_converted)
    median_val = calculate_median(numeric_values)
    if median_val is None:
        return imputed_data

    for r_idx, row in enumerate(imputed_data):
        if len(row) > col_idx:
            val = row[col_idx]
            is_missing = False
            if is_categorical_converted and val == -1:
                is_missing = True
            elif isinstance(val, str) and (not val or val.strip() == ''): # Check for empty string
                is_missing = True

            if is_missing:
                imputed_data[r_idx][col_idx] = round(median_val) if is_categorical_converted else f"{median_val:.2f}"
    return imputed_data

# Function to impute with mode (on processed_data)
def impute_with_mode(data, col_idx, is_categorical_converted=False):
    imputed_data = [list(row) for row in data]
    column_values = []
    for row in imputed_data:
        if len(row) > col_idx:
            val = row[col_idx]
            is_missing = False
            if is_categorical_converted and val == -1:
                is_missing = True
            elif isinstance(val, str) and (not val or val.strip() == ''): # Check for empty string
                is_missing = True

            if not is_missing: # Only consider non-missing values for mode calculation
                column_values.append(val)

    mode_vals = calculate_mode(column_values)
    if not mode_vals:
        return imputed_data

    imputation_val = mode_vals[0] # Pick the first mode if multiple exist

    for r_idx, row in enumerate(imputed_data):
        if len(row) > col_idx:
            val = row[col_idx]
            is_missing = False
            if is_categorical_converted and val == -1:
                is_missing = True
            elif isinstance(val, str) and (not val or val.strip() == ''): # Check for empty string
                is_missing = True

            if is_missing:
                imputed_data[r_idx][col_idx] = imputation_val # Impute with the mode value
    return imputed_data


# Streamlit App
st.title("Heart Disease Prediction Dataset Viewer")

# File path
FILE_PATH = "/content/Heart_Disease_Prediction.csv"

headers, raw_data = parse_csv_data(FILE_PATH)

if not headers or not raw_data:
    st.warning("No data to display or an error occurred during parsing.")
else:
    st.write(f"Dataset loaded: `{FILE_PATH}`")
    st.write(f"Total records: {len(raw_data)}")

    # Specify categorical columns to convert
    categorical_columns_to_convert = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

    # Convert categorical data to numerical
    processed_data = convert_categorical_to_numerical(headers, raw_data, categorical_columns_to_convert)

    # --- Sorting Options ---
    st.sidebar.header("Sorting Options")
    sort_column = st.radio("Sort by column", options=headers)
    sort_ascending = st.checkbox("Ascending order", value=True)

    sorted_data = []
    if sort_column:
        # To sort, we need the index of the column
        try:
            sort_col_idx = headers.index(sort_column)
            # Create a sorting key function
            def get_sort_key_for_list(row_list):
                value = row_list[sort_col_idx]
                try:
                    return float(value) # Try to sort numerically if possible
                except ValueError:
                    return value # Otherwise, sort as string
            sorted_data = sorted(processed_data, key=get_sort_key_for_list, reverse=not sort_ascending)
        except ValueError:
            sorted_data = processed_data # Fallback if column not found
    else:
        sorted_data = processed_data # No sort column selected


    # --- Pagination Options ---
    st.sidebar.header("Pagination Options")
    items_per_page_options = [10, 25, 50, 100]
    items_per_page = st.selectbox("Items per page", options=items_per_page_options, index=0)

    total_pages = math.ceil(len(sorted_data) / items_per_page)

    # Using session state for current page
    if 'current_page' not in st.session_state:
        st.session_state.current_page = 1

    col1, col2, col3 = st.sidebar.columns([1,2,1])
    with col1:
        if st.button("Previous Page", key="prev_page"):
            if st.session_state.current_page > 1:
                st.session_state.current_page -= 1
    with col3:
        if st.button("Next Page", key="next_page"):
            if st.session_state.current_page < total_pages:
                st.session_state.current_page += 1
    with col2:
        st.write(f"Page {st.session_state.current_page} of {total_pages}")


    start_idx = (st.session_state.current_page - 1) * items_per_page
    end_idx = start_idx + items_per_page
    paginated_data = sorted_data[start_idx:end_idx]

    # --- Display Data ---
    st.header("Dataset Preview")
    st.table([headers] + paginated_data)

    # --- Summary Statistics Options ---
    st.sidebar.header("Summary Statistics")
    selected_stat_column = st.sidebar.selectbox("Select column for statistics", options=[''] + headers, index=0)

    if selected_stat_column:
        st.subheader(f"Summary Statistics for '{selected_stat_column}'")
        col_idx = headers.index(selected_stat_column)
        column_data = [row[col_idx] for row in processed_data if len(row) > col_idx]

        # Filter out non-numeric values for calculations that require them
        numeric_column_data = []
        for x in column_data:
            try:
                numeric_column_data.append(float(x))
            except ValueError:
                continue

        mean_val = calculate_mean(column_data)
        median_val = calculate_median(column_data)
        mode_val = calculate_mode(column_data)
        min_val = calculate_min(column_data)
        max_val = calculate_max(column_data)
        std_dev_val = calculate_std_dev(column_data)

        stats_data = {
            "Statistic": ["Mean", "Median", "Mode", "Min", "Max", "Standard Deviation", "Count"],
            "Value": [
                f"{mean_val:.2f}" if mean_val is not None else "N/A",
                f"{median_val:.2f}" if median_val is not None else "N/A",
                str(mode_val) if mode_val is not None else "N/A",
                f"{min_val:.2f}" if min_val is not None else "N/A",
                f"{max_val:.2f}" if max_val is not None else "N/A",
                f"{std_dev_val:.2f}" if std_dev_val is not None else "N/A",
                len(numeric_column_data)
            ]
        }
        st.table(stats_data)


Writing app.py


In [3]:
# Re-defining utility functions for clarity within this demonstration context
import csv
import math

# Function to manually parse CSV without pandas
def parse_csv_data(filepath):
    data = []
    headers = []
    try:
        with open(filepath, 'r', newline='', encoding='utf-8') as f:
            reader = csv.reader(f)
            headers = next(reader) # First row is headers
            for row in reader:
                if row: # Ensure row is not empty
                    data.append(row)
    except FileNotFoundError:
        print(f"File not found: {filepath}")
        return [], []
    except Exception as e:
        print(f"Error parsing CSV: {e}")
        return [], []
    return headers, data

# Function to convert specified categorical columns to numerical
def convert_categorical_to_numerical(headers, data, categorical_cols):
    processed_data = [list(row) for row in data]
    converted_cols_names = [] # Keep track of which columns were converted

    for col_name in categorical_cols:
        try:
            col_idx = headers.index(col_name)
            converted_cols_names.append(col_name) # Mark as converted
            unique_values = sorted(list(set(row[col_idx] for row in processed_data if len(row) > col_idx and row[col_idx].strip() != '')))
            mapping = {value: i for i, value in enumerate(unique_values)}

            for i, row in enumerate(processed_data):
                if len(row) > col_idx:
                    original_value = row[col_idx]
                    if original_value.strip() == '': # Treat empty string as missing for categorical before mapping
                        processed_data[i][col_idx] = -1
                    else:
                        processed_data[i][col_idx] = mapping.get(original_value, -1)
        except ValueError:
            print(f"Categorical column '{col_name}' not found in dataset headers. Skipping conversion.")
    return processed_data, converted_cols_names

# Function to calculate mean
def calculate_mean(data_list):
    if not data_list:
        return None
    total = 0
    count = 0
    for x in data_list:
        try:
            total += float(x)
            count += 1
        except ValueError:
            continue # Skip non-numeric values
    return total / count if count > 0 else None

# Function to calculate median
def calculate_median(data_list):
    numeric_data = []
    for x in data_list:
        try:
            numeric_data.append(float(x))
        except ValueError:
            continue
    if not numeric_data:
        return None
    sorted_data = sorted(numeric_data)
    n = len(sorted_data)
    if n % 2 == 1:
        return sorted_data[n // 2]
    else:
        mid1 = sorted_data[n // 2 - 1]
        mid2 = sorted_data[n // 2]
        return (mid1 + mid2) / 2

# Function to calculate mode
def calculate_mode(data_list):
    counts = {}
    for x in data_list:
        counts[x] = counts.get(x, 0) + 1
    if not counts:
        return None
    max_count = 0
    modes = []
    for value, count in counts.items():
        if count > max_count:
            max_count = count
            modes = [value]
        elif count == max_count and value not in modes:
            modes.append(value)
    return modes

# Function to identify missing values
def identify_missing_values(headers, data, converted_categorical_cols_names):
    missing_counts = {header: 0 for header in headers}
    missing_data_info = [] # List of (row_idx, col_name) for missing values

    for row_idx, row in enumerate(data):
        for col_idx, value in enumerate(row):
            col_name = headers[col_idx]
            is_missing = False

            if col_name in converted_categorical_cols_names and value == -1:
                is_missing = True
            elif isinstance(value, str) and (not value or value.strip() == ''):
                is_missing = True

            if is_missing:
                missing_counts[col_name] += 1
                missing_data_info.append((row_idx, col_name))
    return missing_counts, missing_data_info

# Helper to get numeric data from a column for imputation
def get_column_numeric_values_for_imputation(data, col_idx, is_converted_categorical_col):
    numeric_data = []
    for row in data:
        if len(row) > col_idx:
            val = row[col_idx]
            # Skip if it's a known missing value representation
            if is_converted_categorical_col and val == -1:
                continue
            if isinstance(val, str) and (not val or val.strip() == ''):
                continue
            try:
                numeric_data.append(float(val))
            except ValueError:
                continue # Skip truly non-numeric values that are not empty strings
    return numeric_data

# Function to impute with mean
def impute_with_mean(data_to_impute, col_idx, is_converted_categorical_col=False):
    imputed_data = [list(row) for row in data_to_impute]
    numeric_values = get_column_numeric_values_for_imputation(imputed_data, col_idx, is_converted_categorical_col)
    mean_val = calculate_mean(numeric_values)
    if mean_val is None:
        return imputed_data

    for r_idx, row in enumerate(imputed_data):
        if len(row) > col_idx:
            val = row[col_idx]
            is_missing = False
            if is_converted_categorical_col and val == -1:
                is_missing = True
            elif isinstance(val, str) and (not val or val.strip() == ''):
                is_missing = True

            if is_missing:
                imputed_data[r_idx][col_idx] = round(mean_val) if is_converted_categorical_col else f"{mean_val:.2f}"
    return imputed_data

# Function to impute with median
def impute_with_median(data_to_impute, col_idx, is_converted_categorical_col=False):
    imputed_data = [list(row) for row in data_to_impute]
    numeric_values = get_column_numeric_values_for_imputation(imputed_data, col_idx, is_converted_categorical_col)
    median_val = calculate_median(numeric_values)
    if median_val is None:
        return imputed_data

    for r_idx, row in enumerate(imputed_data):
        if len(row) > col_idx:
            val = row[col_idx]
            is_missing = False
            if is_converted_categorical_col and val == -1:
                is_missing = True
            elif isinstance(val, str) and (not val or val.strip() == ''):
                is_missing = True

            if is_missing:
                imputed_data[r_idx][col_idx] = round(median_val) if is_converted_categorical_col else f"{median_val:.2f}"
    return imputed_data

# Function to impute with mode
def impute_with_mode(data_to_impute, col_idx, is_converted_categorical_col=False):
    imputed_data = [list(row) for row in data_to_impute]
    column_values = []
    for row in imputed_data:
        if len(row) > col_idx:
            val = row[col_idx]
            is_missing = False
            if is_converted_categorical_col and val == -1:
                is_missing = True
            elif isinstance(val, str) and (not val or val.strip() == ''):
                is_missing = True

            if not is_missing:
                column_values.append(val)

    mode_vals = calculate_mode(column_values)
    if not mode_vals:
        return imputed_data

    imputation_val = mode_vals[0] # Pick the first mode if multiple exist

    for r_idx, row in enumerate(imputed_data):
        if len(row) > col_idx:
            val = row[col_idx]
            is_missing = False
            if is_converted_categorical_col and val == -1:
                is_missing = True
            elif isinstance(val, str) and (not val or val.strip() == ''):
                is_missing = True

            if is_missing:
                imputed_data[r_idx][col_idx] = imputation_val
    return imputed_data

# --- Data Loading and Initial Processing ---
FILE_PATH = "/content/Heart_Disease_Prediction.csv"
headers, raw_data = parse_csv_data(FILE_PATH)

categorical_columns_to_convert = ['Sex', 'Chest pain type', 'FBS over 120', 'EKG results', 'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
processed_data, converted_categorical_cols_names = convert_categorical_to_numerical(headers, raw_data, categorical_columns_to_convert)

print("Data preparation complete for imputation demonstration.")

Data preparation complete for imputation demonstration.


In [4]:
print("--- Identifying Missing Values ---")
missing_counts, missing_data_locations = identify_missing_values(headers, processed_data, converted_categorical_cols_names)

total_missing = 0
for col, count in missing_counts.items():
    if count > 0:
        print(f"Column '{col}': {count} missing values")
        total_missing += count

if total_missing == 0:
    print("No missing values found in the dataset after initial processing.")
else:
    print(f"Total missing values across the dataset: {total_missing}")

# Optionally show some rows with missing values
if missing_data_locations:
    print("\nSample rows with missing values (row_idx, column_name):")
    for i in range(min(5, len(missing_data_locations))): # Show first 5 missing locations
        row_idx, col_name = missing_data_locations[i]
        col_idx = headers.index(col_name)
        print(f"  Row {row_idx}, Column '{col_name}': Original value - {raw_data[row_idx][col_idx]}, Processed value - {processed_data[row_idx][col_idx]}")

--- Identifying Missing Values ---
No missing values found in the dataset after initial processing.


In [5]:
print("--- Mean Imputation Demonstration ---")
# Let's pick a numerical column, e.g., 'BP' (Resting Blood Pressure)
# We will create a copy of processed_data to simulate missing values for demonstration if none exist.
imputation_data_mean = [list(row) for row in processed_data]
col_name_mean = 'BP'
col_idx_mean = headers.index(col_name_mean)

# Check if 'BP' is in converted_categorical_cols_names (it shouldn't be, as it's numerical)
is_bp_categorical = col_name_mean in converted_categorical_cols_names

print(f"Demonstrating Mean Imputation for column: '{col_name_mean}'")

# Find a few non-missing values to turn into missing for demo
rows_to_make_missing_mean = []
for i, row in enumerate(imputation_data_mean):
    if len(row) > col_idx_mean and row[col_idx_mean] != '' and row[col_idx_mean] != -1:
        try:
            float(row[col_idx_mean]) # Ensure it's numeric
            rows_to_make_missing_mean.append(i)
            if len(rows_to_make_missing_mean) >= 3: # Simulate 3 missing values
                break
        except ValueError:
            pass

original_values_mean = []
for r_idx in rows_to_make_missing_mean:
    original_values_mean.append(imputation_data_mean[r_idx][col_idx_mean])
    imputation_data_mean[r_idx][col_idx_mean] = '' # Set to empty string for missing

if rows_to_make_missing_mean:
    print(f"Simulated missing values at rows {rows_to_make_missing_mean}. Original values were: {original_values_mean}")

# Perform mean imputation
imputed_data_mean = impute_with_mean(imputation_data_mean, col_idx_mean, is_bp_categorical)

print("\nSample of data before and after Mean Imputation (relevant rows):")
for r_idx in rows_to_make_missing_mean:
    print(f"Row {r_idx}: Before: {imputation_data_mean[r_idx][col_idx_mean]}, After: {imputed_data_mean[r_idx][col_idx_mean]}")

# If no original missing values and no simulated ones (edge case), just print the mean
if not rows_to_make_missing_mean and not missing_counts.get(col_name_mean, 0):
    all_values = get_column_numeric_values_for_imputation(processed_data, col_idx_mean, is_bp_categorical)
    mean_value = calculate_mean(all_values)
    print(f"No missing values found for '{col_name_mean}'. Mean value is: {mean_value:.2f}")

--- Mean Imputation Demonstration ---
Demonstrating Mean Imputation for column: 'BP'
Simulated missing values at rows [0, 1, 2]. Original values were: ['130', '115', '124']

Sample of data before and after Mean Imputation (relevant rows):
Row 0: Before: , After: 131.44
Row 1: Before: , After: 131.44
Row 2: Before: , After: 131.44


In [6]:
print("--- Median Imputation Demonstration ---")
# Let's pick another numerical column, e.g., 'Cholesterol'
imputation_data_median = [list(row) for row in processed_data]
col_name_median = 'Cholesterol'
col_idx_median = headers.index(col_name_median)

is_cholesterol_categorical = col_name_median in converted_categorical_cols_names

print(f"Demonstrating Median Imputation for column: '{col_name_median}'")

# Simulate a few missing values if the column is clean for demonstration purposes
rows_to_make_missing_median = []
for i, row in enumerate(imputation_data_median):
    if len(row) > col_idx_median and row[col_idx_median] != '' and row[col_idx_median] != -1:
        try:
            float(row[col_idx_median])
            rows_to_make_missing_median.append(i)
            if len(rows_to_make_missing_median) >= 3:
                break
        except ValueError:
            pass

original_values_median = []
for r_idx in rows_to_make_missing_median:
    original_values_median.append(imputation_data_median[r_idx][col_idx_median])
    imputation_data_median[r_idx][col_idx_median] = '' # Set to empty string for missing

if rows_to_make_missing_median:
    print(f"Simulated missing values at rows {rows_to_make_missing_median}. Original values were: {original_values_median}")

# Perform median imputation
imputed_data_median = impute_with_median(imputation_data_median, col_idx_median, is_cholesterol_categorical)

print("\nSample of data before and after Median Imputation (relevant rows):")
for r_idx in rows_to_make_missing_median:
    print(f"Row {r_idx}: Before: {imputation_data_median[r_idx][col_idx_median]}, After: {imputed_data_median[r_idx][col_idx_median]}")

if not rows_to_make_missing_median and not missing_counts.get(col_name_median, 0):
    all_values = get_column_numeric_values_for_imputation(processed_data, col_idx_median, is_cholesterol_categorical)
    median_value = calculate_median(all_values)
    print(f"No missing values found for '{col_name_median}'. Median value is: {median_value:.2f}")

--- Median Imputation Demonstration ---
Demonstrating Median Imputation for column: 'Cholesterol'
Simulated missing values at rows [0, 1, 2]. Original values were: ['322', '564', '261']

Sample of data before and after Median Imputation (relevant rows):
Row 0: Before: , After: 244.00
Row 1: Before: , After: 244.00
Row 2: Before: , After: 244.00


In [7]:
print("--- Mode Imputation Demonstration ---")
# Let's pick a categorical column that was converted to numerical, e.g., 'Number of vessels fluro' (number of major vessels (0-3) colored by flourosopy)
imputation_data_mode = [list(row) for row in processed_data]
col_name_mode = 'Number of vessels fluro'
col_idx_mode = headers.index(col_name_mode)

is_vessels_categorical = col_name_mode in converted_categorical_cols_names # This should be True

print(f"Demonstrating Mode Imputation for column: '{col_name_mode}'")

# Simulate a few missing values (-1) for demonstration purposes
rows_to_make_missing_mode = []
for i, row in enumerate(imputation_data_mode):
    if len(row) > col_idx_mode and row[col_idx_mode] != -1: # Ensure it's not already missing
        rows_to_make_missing_mode.append(i)
        if len(rows_to_make_missing_mode) >= 3: # Simulate 3 missing values
            break

original_values_mode = []
for r_idx in rows_to_make_missing_mode:
    original_values_mode.append(imputation_data_mode[r_idx][col_idx_mode])
    imputation_data_mode[r_idx][col_idx_mode] = -1 # Set to -1 for missing categorical

if rows_to_make_missing_mode:
    print(f"Simulated missing values at rows {rows_to_make_missing_mode}. Original values were: {original_values_mode}")

# Perform mode imputation
imputed_data_mode = impute_with_mode(imputation_data_mode, col_idx_mode, is_vessels_categorical)

print("\nSample of data before and after Mode Imputation (relevant rows):")
for r_idx in rows_to_make_missing_mode:
    print(f"Row {r_idx}: Before: {imputation_data_mode[r_idx][col_idx_mode]}, After: {imputed_data_mode[r_idx][col_idx_mode]}")

if not rows_to_make_missing_mode and not missing_counts.get(col_name_mode, 0):
    all_values = get_column_numeric_values_for_imputation(processed_data, col_idx_mode, is_vessels_categorical)
    mode_value = calculate_mode(all_values)
    print(f"No missing values found for '{col_name_mode}'. Mode value(s) is: {mode_value}")

--- Mode Imputation Demonstration ---
Demonstrating Mode Imputation for column: 'Number of vessels fluro'
Simulated missing values at rows [0, 1, 2]. Original values were: [3, 0, 0]

Sample of data before and after Mode Imputation (relevant rows):
Row 0: Before: -1, After: 0
Row 1: Before: -1, After: 0
Row 2: Before: -1, After: 0


In [22]:
print('All imputation demonstrations are complete. Please review the output from the previous cells.')

All imputation demonstrations are complete. Please review the output from the previous cells.


## Report: Multiple Imputation Techniques

Missing data is a common challenge in real-world datasets, and how it's handled can significantly impact analysis results. Multiple imputation involves replacing missing values with substituted values. Here, we discuss three common manual imputation techniques: Mean, Median, and Mode imputation.

### 1. Mean Imputation

**Method:** In mean imputation, missing values in a numerical column are replaced by the mean (average) of the observed values in that same column.

**When to use:**
*   **Numerical data:** This technique is suitable only for numerical features.
*   **Missing Completely At Random (MCAR) or Missing At Random (MAR):** It's most appropriate when data is missing randomly, and the missingness does not depend on the values of the variable itself or other observed variables.
*   **Small percentage of missing data:** When only a small fraction of data is missing, mean imputation can be a simple and effective solution.
*   **Minimal impact on mean:** It preserves the mean of the column, which can be useful if the mean is a critical statistic for your analysis.

**Pros:**
*   Simple to implement and computationally efficient.
*   Maintains the overall mean of the variable.

**Cons:**
*   **Reduces variance:** It can artificially reduce the variability of the data, as all imputed values are the same.
*   **Distorts relationships:** Can weaken correlations between the imputed variable and other variables.
*   **Sensitive to outliers:** The mean is sensitive to outliers, so extreme values can skew the imputed value.
*   **Does not reflect uncertainty:** It doesn't account for the uncertainty associated with the imputed values.

### 2. Median Imputation

**Method:** Similar to mean imputation, but missing numerical values are replaced by the median of the observed values in the column.

**When to use:**
*   **Numerical data:** Applicable for numerical features.
*   **Skewed distributions or presence of outliers:** The median is a robust measure of central tendency, less affected by extreme values than the mean. Therefore, it's preferred when the data distribution is skewed or contains significant outliers.
*   **Small percentage of missing data:** Like mean imputation, it's best for a small proportion of missing data.

**Pros:**
*   Robust to outliers and skewed data distributions.
*   Simple to implement.

**Cons:**
*   Still reduces variance and can distort relationships, similar to mean imputation.
*   Does not reflect uncertainty.

### 3. Mode Imputation

**Method:** Missing values in a column are replaced by the mode (most frequently occurring value) of the observed values in that column.

**When to use:**
*   **Categorical or discrete numerical data:** This is the most appropriate method for categorical variables (nominal or ordinal) and discrete numerical variables.
*   **Any missing data pattern (MCAR/MAR):** It can be used when the missingness is random or even non-random, but the mode is a reasonable representation.
*   **Small percentage of missing data:** Best for small amounts of missing data to avoid introducing significant bias.

**Pros:**
*   Suitable for categorical and discrete data types.
*   Simple to implement.
*   Preserves the most common category/value.

**Cons:**
*   Can artificially inflate the frequency of the mode category/value.
*   Doesn't consider the relationships between variables.
*   If there are multiple modes, a choice must be made (e.g., picking the first one), which can be arbitrary.
*   Does not reflect uncertainty.

### Justification of Chosen Method(s):

For this demonstration:
*   **Mean Imputation** was applied to `trestbps` (Resting Blood Pressure), a continuous numerical variable. This choice is often made for numerical data that is approximately symmetrically distributed and without severe outliers, as it preserves the column's mean.
*   **Median Imputation** was applied to `chol` (Cholesterol), another continuous numerical variable. This is a good alternative for numerical data, especially if `chol` is found to have a skewed distribution or outliers, as the median is less sensitive to such anomalies.
*   **Mode Imputation** was applied to `ca` (Number of major vessels (0-3) colored by flourosopy), which represents a discrete, ordinal categorical feature (converted to numerical values 0, 1, 2, 3). Mode imputation is the natural choice for such variables, as it replaces missing values with the most frequent category, maintaining the discrete nature of the data.

Finally, I'll run the Streamlit app. After execution, a public URL will be generated. Click on that URL to interact with the Streamlit app.

In [8]:
import subprocess
import time

!pip install pyngrok streamlit

from pyngrok import ngrok

# Terminate any running Streamlit processes
!pkill -f streamlit

# Start Streamlit app in the background
streamlit_process = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.enableCORS=False",
    "--server.enableXsrfProtection=False",
    "--browser.gatherUsageStats=False"
])

time.sleep(5) # Give Streamlit a moment to start

# Authenticate ngrok. Replace 'YOUR_NGROK_AUTHTOKEN' with your actual token if needed.
ngrok.set_auth_token("3FffIo4dyhAPhgAKHzBgJmWqW9H_81M6pUJLLN58mBeBtbWRL")

# Open a ngrok tunnel to the Streamlit port (8501)
public_url = ngrok.connect(8501)
print(f"Streamlit App URL: {public_url}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 100.6 MB/s eta 0:00:00
Streamlit App URL: NgrokTunnel: "https://cubicle-humility-pried.ngrok-free.dev" -> "http://localhost:8501"
